In [ ]:
import os
os.makedirs('figures', exist_ok=True)

Notebooks to analyse the survey conducted between December 2025-April 2026 within the interdisciplinary AHOI Project.

Use this notebook for the evaluation of the pre and post-survey questionnaires (Likert scale).
Use the other notebooks for correlation analysis and the sentiment analysis on the open questions.

The survey outline and proposed research has been published here:
https://link.springer.com/chapter/10.1007/978-3-032-19099-4_8

The evaluation paper will be presented at the ECML-PKDD conference in September 2026 and added on github.

**If you find the scripts helpful and/or use them in your own research, please cite the paper and the github repository. Thank you.**

In [ ]:
# Import first necessary libraries for data handling, statistics, and plotting.
# Each cell contains also individual libs to comment in if only specific part is needed.
# Some cells also contain optional print commands to check if you are actually getting the correct data and dimensions

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Change file name here
df = pd.read_csv('/your/file.csv')

# Optional print cmds to check dimensions and first db entries
#print(f"Original DataFrame shape: {df.shape}")
#print("Original DataFrame head:")
#display(df.head())#
#print(df.columns)

Evaluation ATAS: Abbreviated Technology Anxiety Scale. Used as a pre-questionnaire.

Extract only the ATAS responses (5-point Likert scale); includes data cleaning (e.g. removal of strings, NaN removal, etc.)

In [ ]:
# Select only the 'ATAS' columns (pre-questionnaire)

atas_columns = [col for col in df.columns if 'ATAS' in col]
df_atas = df[atas_columns].copy()

# Optional print for dimension check and first db entries
#print(f"DataFrame with ATAS columns shape: {df_atas.shape}")
#print("ATAS DataFrame head:")
#display(df_atas.head())

# Convert all ATAS columns to numeric, coercing errors to NaN
for col in df_atas.columns:
    df_atas[col] = pd.to_numeric(df_atas[col], errors='coerce')

# Remove rows where all ATAS columns are NaN (these rows likely contained only text)
df_atas_cleaned = df_atas.dropna(how='all')

# Keep only values between 1 and 5 (inclusive)
df_atas_cleaned = df_atas_cleaned[(df_atas_cleaned >= 1).all(axis=1) & (df_atas_cleaned <= 5).all(axis=1)]

# Reset index
df_atas_cleaned = df_atas_cleaned.reset_index(drop=True)

# Save the new dataframe to a new csv file
output_file_name = 'ATAS.csv'
df_atas_cleaned.to_csv(output_file_name, index=False)

# Optional print for dimension check and first db entries
#print(f"Cleaned ATAS DataFrame shape: {df_atas_cleaned.shape}")
#print("Cleaned ATAS DataFrame head:")
display(df_atas_cleaned.head())
#print(f"Cleaned data saved to '{output_file_name}'")

Calculate desired (descriptive) statistics.

In [ ]:
# Calculate (descriptive) statistics
stats = df_atas_cleaned.describe().T
stats['median'] = df_atas_cleaned.median()
stats = stats[['mean', 'median', 'std', 'min', 'max']]

print("Descriptive Statistics for ATAS Likert Items:")
display(stats)

Plot ATAS results in a bar chart.

In [ ]:
# Plot response distribution (good for first check but difficult to read).
# Comment in the libraries if only plotting needed
#import matplotlib.pyplot as plt
#import seaborn as sns

plt.figure(figsize=(12, 6))
df_melted = df_atas_cleaned.melt(var_name='Item', value_name='Score')

sns.countplot(data=df_melted, x='Item', hue='Score', palette='viridis')
plt.title('Distribution of Likert Scale Responses per ATAS Item')
plt.xlabel('ATAS Question')
plt.ylabel('Frequency')
plt.legend(title='Score (1-5)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Creates bar chart per item with median and std (mean just for reference as Likert is ordinal data)
# Comment-in import command if used as stand-alone
#import matplotlib.pyplot as plt
#import seaborn as sns
#import pandas as pd
#import numpy as np

# Calculate means, standard deviations, and medians
means = df_atas_cleaned.mean()
stds = df_atas_cleaned.std()
medians = df_atas_cleaned.median()

plt.figure(figsize=(10, 6))

# Create the bar plot with whiskers indicating the standard deviation (sd)
sns.barplot(data=df_atas_cleaned, palette='viridis', errorbar='sd', capsize=0.1)

# Add median and mean values for reference
plt.scatter(x=range(len(medians)), y=medians, color='blue', zorder=5, label='Median', marker='o')
plt.scatter(x=range(len(means)), y=means, color='red', marker='*', s=100, zorder=6, label='Mean')

plt.title('Mean and Median ATAS Scores')
plt.xlabel('ATAS Items')
plt.ylabel('Likert Scale (1-5)')
plt.ylim(1, 5)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

Response distribution for each ATAS item individually to see where the spread is coming from.

In [ ]:
#import matplotlib.pyplot as plt
#import seaborn as sns
import math

num_cols = 3
num_rows = math.ceil(len(df_atas_cleaned.columns) / num_cols)

fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 4 * num_rows), sharey=True)
axes = axes.flatten()

# Plot frequency counts for each column
for i, col in enumerate(df_atas_cleaned.columns):
    sns.countplot(x=df_atas_cleaned[col], ax=axes[i], palette='viridis', hue=df_atas_cleaned[col], legend=False)
    axes[i].set_title(f'Frequency: {col}')
    axes[i].set_xlabel('Likert Score')
    axes[i].set_ylabel('Count')
    axes[i].set_xticks(range(0, 5))
    axes[i].set_xticklabels(['1', '2', '3', '4', '5'])

# Remove any empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

Descriptive statistics of TIA and altered Hoffman scale for 'Scenario 1'.
Includes data cleaning, reverse-coding, simple statistics computation, and visualization (plotting) of data.

Trust in Automation (TIA) post-questionnaire after presentation of 'Scenario1' with the XAI navigation assistant.

TIA needs reverse-coding of items 6-11. If not sure whether items need reverse-coding, use correlation plots for negative correlations or try Cronbach's alpha.

In [ ]:
def cronbach_alpha(df):
    k = df.shape[1]
    item_vars = df.var(axis=0, ddof=1).sum()
    total_score_var = df.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - (item_vars / total_score_var))

In [ ]:
# Extract columns containing 'Scenario1_TIA' (change column label if not matching your input)
tia_cols = [col for col in df.columns if 'Scenario1_TIA' in col]
df_tia = df[tia_cols].copy()

# Preprocessing: Convert to numeric and keep only 1-5 range
for col in df_tia.columns:
    df_tia[col] = pd.to_numeric(df_tia[col], errors='coerce')

# Drop rows that are entirely NaN
df_tia_cleaned = df_tia.dropna(how='all')

# Filter for valid Likert range 1-5
df_tia_cleaned = df_tia_cleaned[(df_tia_cleaned >= 1).all(axis=1) & (df_tia_cleaned <= 5).all(axis=1)]
df_tia_cleaned.to_csv('Scenario1_TIA.csv', index=False)

#Rename columns for x-axis readability
plot_df = df_tia_cleaned.copy()
plot_df.columns = [f'TIA_{i+1}' for i in range(len(plot_df.columns))]

df_tia_reversed = plot_df.copy()

# Reverse-code items TIA_6 through TIA_11 (assuming 1-5 scale, 6 - value)
reverse_cols = ['TIA_6', 'TIA_7', 'TIA_8', 'TIA_9', 'TIA_10', 'TIA_11']
for col in reverse_cols:
    df_tia_reversed[col] = 6 - df_tia_reversed[col]

# Calculate Cronbach's alpha on the responses with reverse-coding
cronbach_tia = cronbach_alpha(df_tia_reversed)
print(f"Cronbach's Alpha for the TIA Items Scenario 1 (after reverse-coding): {cronbach_tia:.3f}")

In [ ]:
# Calculate statistics for the reverse-coded TIA items and plot results
# Creates bar chart per item with median and std (mean just for reference)
tia_rev_means = df_tia_reversed.mean()
tia_rev_medians = df_tia_reversed.median()
tia_rev_stds = df_tia_reversed.std()

# Visualization
plt.figure(figsize=(14, 6))
sns.barplot(data=df_tia_reversed, palette='viridis', errorbar='sd', capsize=0.1)

# Add median and mean markers
plt.scatter(x=range(len(tia_rev_medians)), y=tia_rev_medians, color='blue', zorder=5, label='Median', marker='o')
plt.scatter(x=range(len(tia_rev_means)), y=tia_rev_means, color='red', marker='*', s=100, zorder=6, label='Mean')

plt.title('Mean and Median Scenario 1 TIA Scores (Items 6-11 Reverse-Coded)')
plt.xlabel('TIA Items')
plt.ylabel('Likert Scale (1-5)')
plt.ylim(1, 5.5)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Display statistics table (comment-in if desired)
#tia_rev_stats = df_tia_reversed.describe().T[['mean', '50%', 'std']]
#tia_rev_stats.columns = ['Mean', 'Median', 'Std Dev']
#display(tia_rev_stats)

# **EVALUATIONS SCENARIO 1**

Evaluation of Trust in Automation (TIA) scale (5-point Likert scale, presented after scenario 1). **Reverse-coding needed for items 6-11!**
TIA-related data is parsed, cleaned, reverse-coded, and visualized based on descriptive statistics and item-level evaluation.

In [ ]:
# helper function to check internal consistency of scale
def cronbach_alpha(df):
    k = df.shape[1]
    item_vars = df.var(axis=0, ddof=1).sum()
    total_score_var = df.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - (item_vars / total_score_var))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_cleaned = df.iloc[2:].copy()

scenario1_tia_cols = [col for col in df_cleaned.columns if 'Scenario1_TIA_' in col]

df_tia = df_cleaned[scenario1_tia_cols].copy()

for col in df_tia.columns:
    df_tia[col] = pd.to_numeric(df_tia[col], errors='coerce')

df_tia.dropna(inplace=True)

#print(f"Shape of TIA data after cleaning: {df_tia.shape}")
#display(df_tia.head())

reverse_code_cols = [f'Scenario1_TIA_{i}' for i in range(6, 12)]

for col in reverse_code_cols:
    if col in df_tia.columns:
        df_tia[col] = 6 - df_tia[col]
        print(f"Reverse coded column: {col}")

#display(df_tia.head())

In [ ]:
from matplotlib.lines import Line2D

df_melted = df_tia.melt(var_name='Item', value_name='Response')

item_mapping = {f'Scenario1_TIA_{i}': f'Item {i}' for i in range(1, 12)}
df_melted['Item_Label'] = df_melted['Item'].map(item_mapping)

fig = plt.figure(figsize=(15, 8))

sns.barplot(
    data=df_melted,
    x='Item_Label',
    y='Response',
    estimator=np.mean,
    palette='viridis',
    errorbar='sd',
    capsize=0.2,
    hue='Item_Label',
    legend=False
)

mean_responses = df_melted.groupby('Item_Label')['Response'].mean().loc[item_mapping.values()]
median_responses = df_melted.groupby('Item_Label')['Response'].median().loc[item_mapping.values()]

x_positions = np.arange(len(item_mapping))

plt.scatter(
    x=x_positions,
    y=mean_responses,
    color='red',
    marker='*',
    s=150,
    label='mean value',
    zorder=3
)

plt.scatter(
    x=x_positions,
    y=median_responses,
    color='black',
    marker='_',
    s=300,
    linewidths=2,
    label='median value',
    zorder=3
)

plt.title('Mean Likert Responses per Scenario1 TIA Item (reverse-coded items 6-11)', fontsize=16)
plt.xlabel('Survey Item', fontsize=12)
plt.ylabel('Mean Likert Response', fontsize=12)
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels if they overlap
plt.yticks(fontsize=10)

# Create a combined legend for error bars, mean asterisks, and median lines
custom_lines = [
    #Line2D([0], [0], color='gray', lw=2, marker='_', markersize=8, label='Mean ± Std Dev'),
    Line2D([0], [0], color='red', marker='*', linestyle='None', markersize=10, label='Mean Value'),
    Line2D([0], [0], color='black', marker='_', linestyle='None', markersize=10, linewidth=2, label='Median Value')
]
plt.legend(handles=custom_lines, title='', loc='upper left')

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

Item-level respinse frequency plot for TIA scale.

In [ ]:
# Response frequency per item

g = sns.catplot(
    data=df_melted,
    x='Response',
    col='Item_Label',
    col_wrap=4,
    kind='count',
    palette='viridis',
    height=4, aspect=1,
    hue='Response',
    legend=False
)

g.set_axis_labels('Likert Response', 'Frequency')
g.set_titles('Item: {col_name}')
g.fig.suptitle('Frequency Distribution of Likert Responses per Item (reverse-coded items 6-11)', y=1.02, fontsize=16) # Adjust main title position

#plt.tight_layout()
#plt.show()

Evaluation of altered Hoffman scale (5-point Likert scale, presented after scenario 1). **Reverse-coding needed for items 1,2,5, 7!**
Hoffman-related data is parsed, cleaned, reverse-coded, and visualized based on descriptive statistics and item-level evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Processing raw data
df_cleaned_headers = df.iloc[2:].copy()
df_cleaned_headers.reset_index(drop=True, inplace=True)

quest_cols = [col for col in df_cleaned_headers.columns if 'Scenario1_Quest' in col]
df_quest = df_cleaned_headers[quest_cols].copy()

for col in df_quest.columns:
    df_quest[col] = pd.to_numeric(df_quest[col], errors='coerce')

df_quest_cleaned = df_quest.dropna(how='all')
if not df_quest_cleaned.empty:
    df_quest_cleaned = df_quest_cleaned[(df_quest_cleaned >= 1).all(axis=1) & (df_quest_cleaned <= 5).all(axis=1)]
df_quest_cleaned = df_quest_cleaned.reset_index(drop=True)

plot_df_quest = df_quest_cleaned.copy()
plot_df_quest.columns = [f'Quest_{i+1}' for i in range(len(plot_df_quest.columns))]

# reverse-code items 1, 2, 5, 7
df_quest_reversed = plot_df_quest.copy()
reverse_indices = [1, 2, 5, 7]
reverse_cols_quest_names = [f'Quest_{i}' for i in reverse_indices]

for col in reverse_cols_quest_names:
    df_quest_reversed[col] = 6 - df_quest_reversed[col]

# calculate descriptive statistics
quest_rev_means = df_quest_reversed.mean()
quest_rev_medians = df_quest_reversed.median()
quest_rev_stds = df_quest_reversed.std()

cronbach_s1 = cronbach_alpha(df_quest_reversed)
print(f"Cronbach's alpha Scenario 1 (Hoffman Items 1,2,5,7 Reversed): {cronbach_s1:.3f}")

# Visualization
plt.figure(figsize=(14, 6))
sns.barplot(data=df_quest_reversed, palette='viridis', errorbar='sd', capsize=0.1)
plt.scatter(x=range(len(quest_rev_medians)), y=quest_rev_medians, color='blue', zorder=5, label='Median', marker='o')
plt.scatter(x=range(len(quest_rev_means)), y=quest_rev_means, color='red', marker='*', s=100, zorder=6, label='Mean')

plt.title('Mean and Median Scenario 1 Hoffmann Scores (Items 1,2,5,7 reverse-coded) - Altered Hoffman Scale', fontsize=16)
plt.xlabel('Hoffman Items', fontsize=14)
plt.ylabel('Likert Scale (1-5)', fontsize=14)
plt.ylim(1, 5.5)
plt.legend(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)

num_quest_items = len(df_quest_reversed.columns)
plt.xticks(ticks=range(num_quest_items), labels=[f'Item {i+1}' for i in range(num_quest_items)], fontsize=12)
plt.yticks(fontsize=12)

**Item-level and factor analysis on altered Hoffman scale (Scenario 1).**


In [ ]:
print("Mean Scores of Scenario 1 Hoffman Items (after reverse-coding):")
display(df_quest_reversed.mean().to_frame(name='Mean Score'))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get the mean scores (already calculated as quest_rev_means)
mean_scores = df_quest_reversed.mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=mean_scores.index, y=mean_scores.values, palette='viridis', hue=mean_scores.index, legend=False)

plt.title('Mean Scores of Scenario 1 Hoffman Items (Reverse-Coded)')
plt.xlabel('Hoffman Items')
plt.ylabel('Mean Likert Score (1-5)')
plt.ylim(0, 5) # Likert scale is 1-5
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Individual Response Distributions for Scenario 1, altered Hoffman scale**

In [ ]:
#import pandas as pd
#import numpy as np
#import matplotlib.pyplot as plt
#import seaborn as sns
import math

print("\n--- Item-level Descriptive Analysis for Scenario 1 Hoffman Scale (df_quest_reversed) ---")

# Calculate comprehensive descriptive statistics for each item (includes count, mean, std, min, 25%, 50% (median), 75%, max)
quest_item_stats = df_quest_reversed.describe().T

# Add skewness and kurtosis for more detailed distribution understanding
quest_item_stats['skewness'] = df_quest_reversed.skew()
quest_item_stats['kurtosis'] = df_quest_reversed.kurt()

print("Descriptive Statistics for each Hoffman Scale Item:")
display(quest_item_stats)

# Plot individual response distributions for each Hoffman item
num_cols = 3 # Number of columns for the subplots
num_items = len(df_quest_reversed.columns)
num_rows = math.ceil(num_items / num_cols)

fig, axes = plt.subplots(num_rows, num_cols, figsize=(5 * num_cols, 4 * num_rows), sharey=True)
axes = axes.flatten() # Flatten the array of axes for easy iteration

print("\nIndividual Response Distributions for Hoffman Scale Items:")
for i, col in enumerate(df_quest_reversed.columns):
    sns.countplot(x=df_quest_reversed[col], ax=axes[i], palette='viridis', hue=df_quest_reversed[col], legend=False)
    axes[i].set_title(f'Frequency: Item {i+1} - Scenario 1 Hoffman (altered)')
    axes[i].set_xlabel('Likert Score')
    axes[i].set_ylabel('Count')
    # Ensure xticks are set for all possible Likert scores (1-5)
    axes[i].set_xticks(range(1, 6)) # Set ticks from 1 to 5
    axes[i].set_xticklabels(['1', '2', '3', '4', '5'])

# Remove any empty subplots if the number of items is not a perfect multiple of 'num_cols'
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### Item-Total Correlations

Item-total correlations measure the correlation of each item with the total score of the scale. Items with low or negative item-total correlations might not be measuring the same construct as the rest of the scale, contributing to low internal consistency (Cronbach's Alpha).

In [ ]:
# Calculate the total score for each respondent across all items
total_score = df_quest_reversed.sum(axis=1)

item_total_correlations = {}

for col in df_quest_reversed.columns:
    score_excluding_item = total_score - df_quest_reversed[col]
    correlation = df_quest_reversed[col].corr(score_excluding_item) # Pearson correlation
    item_total_correlations[col] = correlation

# Conversion pandas Series (for viz.)
item_total_correlations_series = pd.Series(item_total_correlations, name='Item-Total Correlation')

print("Item-Total Correlations for Scenario 1 Hoffman Scale (after reverse-coding):")
display(item_total_correlations_series.sort_values(ascending=False))

**Exploratory Factor Analysis (EFA) for Scenario 1 Hoffman (altered) scale**

---



Given the low Cronbach's alpha and some low/negative item-total correlations, it is possible that the 8 Hoffman scale items are not measuring a single, unidimensional construct. Therefore, we use "Exploratory Factor Analysis" (EFA) to identify latent variables (factor loadings).

In [ ]:
# Exploratory Factor Analysis (EFA) for Scenario 1 (short)
# Use the data frame "df_quest_reversed" (Items 1, 2, 5, 7 reversed); change variable name if necessary!
!pip install factor_analyzer
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

print("--- EFA for Scenario 1 (Items 1,2,5,7 Reversed) ---")
kmo_all, kmo_model = calculate_kmo(df_quest_reversed)
bartlett_p = calculate_bartlett_sphericity(df_quest_reversed)[1]
print(f"KMO: {kmo_model:.3f}, Bartlett p-value: {bartlett_p:.3f}")

fa = FactorAnalyzer(rotation='varimax')
fa.fit(df_quest_reversed)
ev, v = fa.get_eigenvalues()
print(f"Eigenvalues: {ev}")

# Display Loadings
loadings = pd.DataFrame(fa.loadings_, index=df_quest_reversed.columns, columns=[f'Factor {i+1}' for i in range(fa.loadings_.shape[1])])
display(loadings)

In [ ]:
# Exploratory Factor Analysis (EFA) for Scenario 1 (long; includes scree-plot)
# Use the data frame "df_quest_reversed" (Items 1, 2, 5, 7 reversed); change variable name if necessary!
# Comment-in following code if cell is used individually
'''
!pip install factor_analyzer

import matplotlib.pyplot as plt
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

print("\n--- Exploratory Factor Analysis for Scenario 1 Hoffman Scale ---")
'''

kmo_all_hoff_s1, kmo_model_hoff_s1 = calculate_kmo(df_quest_reversed)
print(f"KMO Test (Scenario 1 Hoffman): {kmo_model_hoff_s1:.3f}")

# Bartlett's Test of Sphericity: tests if the correlation matrix is an identity matrix (i.e., items are unrelated)
# A p-value < 0.05 suggests the data is suitable for factor analysis.
bartlett_p_value_hoff_s1 = calculate_bartlett_sphericity(df_quest_reversed)[1]
print(f"Bartlett's Test p-value (Scenario 1 Hoffman): {bartlett_p_value_hoff_s1:.3f}")

# scree-plot to determine the optimal number of factors using eigenvalues
fa_hoff_s1_scree = FactorAnalyzer(n_factors=df_quest_reversed.shape[1], rotation=None)
fa_hoff_s1_scree.fit(df_quest_reversed)

# get eigenvalues as before
ev_hoff_s1, v_hoff_s1 = fa_hoff_s1_scree.get_eigenvalues()
print(f"Eigenvalues (Scenario 1 Hoffman): {ev_hoff_s1}")

# plot scree plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(ev_hoff_s1) + 1), ev_hoff_s1, marker='o')
plt.axhline(1, color='red', linestyle='--', label='Kaiser Criterion (Eigenvalue = 1)')
plt.title('Scree Plot for Scenario 1 Hoffman Scale Factor Analysis')
plt.xlabel('Factor Number')
plt.ylabel('Eigenvalue')
plt.grid(True)
plt.legend()
plt.show()

# Re-run Factor Analysis with the chosen number of factors using Kaiser Criterion, i.e. keep factors with eigenvalue > 1
n_factors_hoff_s1_final = sum(ev_hoff_s1 > 1)

# handle case when no eigenvalue is > 1
if n_factors_hoff_s1_final == 0:
    n_factors_hoff_s1_final = 1

print(f"\nUsing {n_factors_hoff_s1_final} factor(s) for Scenario 1 Hoffman Scale Factor Analysis (Kaiser Criterion).")

fa_hoff_s1_final = FactorAnalyzer(n_factors=n_factors_hoff_s1_final, rotation='varimax')
fa_hoff_s1_final.fit(df_quest_reversed)

print("\nFactor Loadings for Scenario 1 Hoffman Scale:")
hoff_s1_loadings = pd.DataFrame(fa_hoff_s1_final.loadings_, index=df_quest_reversed.columns)
display(hoff_s1_loadings)

print("\nFactor Variance for Scenario 1 Hoffman Scale:")
hoff_s1_factor_variance = pd.DataFrame(fa_hoff_s1_final.get_factor_variance(),
                                     index=['SS Loadings', 'Proportion Var', 'Cumulative Var'],
                                     columns=[f'Factor {i+1}' for i in range(n_factors_hoff_s1_final)])
display(hoff_s1_factor_variance)

**EVALUATIONS SCENARIO 2**

---



**Data extraction and Analysis for 'Scenario 2' (altered Hoffman, TIA)**

(same data processing and visualizations as for 'Scenario 1')

In [ ]:
scenario2_cols = [col for col in df.columns if 'Scenario2' in col]
scenario2_df = df[scenario2_cols].copy()

print(f"Scenario2 DataFrame shape: {scenario2_df.shape}")
display(scenario2_df.head())

Data cleaning and data extraction.

In [ ]:
# Remove the first two header rows which contain text/metadata
scenario2_df_cleaned = scenario2_df.iloc[2:].copy()

for col in scenario2_df_cleaned.columns:
    scenario2_df_cleaned[col] = pd.to_numeric(scenario2_df_cleaned[col], errors='coerce')

scenario2_df_cleaned = scenario2_df_cleaned.dropna(how='all')

# Filter to keep only values between 1 and 5 (inclusive)
scenario2_df_cleaned = scenario2_df_cleaned[(scenario2_df_cleaned >= 1).all(axis=1) & (scenario2_df_cleaned <= 5).all(axis=1)]
scenario2_df_cleaned = scenario2_df_cleaned.reset_index(drop=True)

# Save the new dataframe to a new csv file
output_file_name_scenario2 = 'Scenario2_cleaned.csv'
scenario2_df_cleaned.to_csv(output_file_name_scenario2, index=False)

''' Uncomment the following lines if check for data content, otherwise leave as comment
print(f"Cleaned Scenario2 DataFrame shape: {scenario2_df_cleaned.shape}")
print("Cleaned Scenario2 DataFrame head:")
display(scenario2_df_cleaned.head())
print(f"Cleaned data saved to '{output_file_name_scenario2}'")

### Reverse-coding TIA and Hoffman (altered) for Scenario 2

In [ ]:
scenario2_df_reversed = scenario2_df_cleaned.copy()

# reverse-code for altered Hoffman scale (items 1,2, 5, 7)
hoff_reverse_indices = [1, 2, 5, 7]
hoff_reverse_cols = [f'Scenario2_Hoff_{i}' for i in hoff_reverse_indices]

for col in hoff_reverse_cols:
    if col in scenario2_df_reversed.columns:
        scenario2_df_reversed[col] = 6 - scenario2_df_reversed[col]

# reverse-code for TIA scale (items 6-11)
tia_reverse_cols = ['Scenario2_TIA_6', 'Scenario2_TIA_7', 'Scenario2_TIA_8', 'Scenario2_TIA_9', 'Scenario2_TIA_10', 'Scenario2_TIA_11']
for col in tia_reverse_cols:
    if col in scenario2_df_reversed.columns:
        scenario2_df_reversed[col] = 6 - scenario2_df_reversed[col]

In [ ]:
# TIA Analysis for Scenario 2
tia_cols_s2 = [col for col in df.columns if 'Scenario2_TIA' in col]
df_tia_s2 = df[tia_cols_s2].iloc[2:].apply(pd.to_numeric, errors='coerce').dropna(how='all')

# Filter valid Likert range 1-5
df_tia_s2 = df_tia_s2[(df_tia_s2 >= 1).all(axis=1) & (df_tia_s2 <= 5).all(axis=1)]
df_tia_s2.columns = [f'TIA_{i+1}' for i in range(len(df_tia_s2.columns))]

# Reverse-coding items 6-11
for col in reverse_cols_tia:
    df_tia_s2[col] = 6 - df_tia_s2[col]

print(f"Cronbach's Alpha Scenario 2 TIA: {cronbach_alpha(df_tia_s2):.3f}")
display(df_tia_s2.describe().T[['mean', '50%', 'std']])

### Desriptive Statistics and Visualizing Reverse-Coded Altered Hoffman scale (Scenario 2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

hoff_cols_scenario2 = [col for col in scenario2_df_reversed.columns if 'Scenario2_Hoff' in col]
df_hoff_scenario2_reversed = scenario2_df_reversed[hoff_cols_scenario2]

hoff_rev_means = df_hoff_scenario2_reversed.mean()
hoff_rev_stds = df_hoff_scenario2_reversed.std()
hoff_rev_medians = df_hoff_scenario2_reversed.median()

plt.figure(figsize=(14, 6))
sns.barplot(data=df_hoff_scenario2_reversed, palette='viridis', errorbar='sd', capsize=0.1)

# Add median and mean markers
plt.scatter(x=range(len(hoff_rev_medians)), y=hoff_rev_medians, color='blue', zorder=5, label='Median', marker='o')
plt.scatter(x=range(len(hoff_rev_means)), y=hoff_rev_means, color='red', marker='*', s=100, zorder=6, label='Mean')

plt.title('Mean and Median Scenario 2 Hoffman Hoffmann Scores (Items 1,2,5,7 Reverse-Coded) - Altered Hoffman Scale', fontsize=16)
plt.xlabel('Hoffman Items', fontsize=14)
plt.ylabel('Likert Scale (1-5)', fontsize=14)
plt.ylim(1, 5.5)
plt.legend(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Relabel x-axis with 'Item 1', 'Item 2', etc.
num_hoff_items = len(df_hoff_scenario2_reversed.columns)
plt.xticks(ticks=range(num_hoff_items), labels=[f'Item {i+1}' for i in range(num_hoff_items)], fontsize=12)
plt.yticks(fontsize=12)

plt.tight_layout()
plt.savefig('Hoffman_Scenario2_Updated_plot.pdf')
plt.show()

cronbach_hoff_s2 = cronbach_alpha(df_hoff_scenario2_reversed)
print(f"Cronbach's Alpha Scenario 2 (Items 1,2,5,7 Reversed): {cronbach_hoff_s2:.3f}")

**Correlation between 'Age' and Scenario 2:**


Examine the correlation between age and the altered Hoffman and TIA scale responses.

In [ ]:
# Prepare 'age' data aligned with cleaned Scenario 2 responses

# temporary data frame with scenario2 columns and age, retaining original indices
temp_df_with_age = df[scenario2_cols + ['age']].copy()

# remove the first two header rows (original index 0 and 1)
temp_df_with_age = temp_df_with_age.iloc[2:].copy()

# convert 'scenario2' columns to numeric and 'age' to numeric
for col in scenario2_cols:
    temp_df_with_age[col] = pd.to_numeric(temp_df_with_age[col], errors='coerce')
temp_df_with_age['age'] = pd.to_numeric(temp_df_with_age['age'], errors='coerce')

scenario2_only_for_dropna = temp_df_with_age[scenario2_cols]
all_scenario2_nan_mask = scenario2_only_for_dropna.isnull().all(axis=1)
temp_df_with_age = temp_df_with_age[~all_scenario2_nan_mask].copy()

# filter to keep only values between 1 and 5 for Scenario2 columns
valid_likert_mask = (temp_df_with_age[scenario2_cols] >= 1).all(axis=1) & \
                    (temp_df_with_age[scenario2_cols] <= 5).all(axis=1)
temp_df_with_age = temp_df_with_age[valid_likert_mask].copy()

# reverse-codiong for Hoffman scale (items 3, 4, 5, 7)
hoff_reverse_cols_full = [col for col in temp_df_with_age.columns if 'Scenario2_Hoff' in col and col in hoff_reverse_cols]
for col in hoff_reverse_cols_full:
    temp_df_with_age[col] = 6 - temp_df_with_age[col]

# identify columns to reverse-code for TIA scale (items 6-11)
tia_reverse_cols_full = [col for col in temp_df_with_age.columns if 'Scenario2_TIA' in col and col in tia_reverse_cols]
for col in tia_reverse_cols_full:
    temp_df_with_age[col] = 6 - temp_df_with_age[col]

df_correlation_final = temp_df_with_age.dropna(subset=['age']).copy()
df_correlation_final['age'] = df_correlation_final['age'].astype(int)

print(f"DataFrame for correlation has {len(df_correlation_final)} rows after cleaning and aligning with 'age'.")
# display(df_correlation_final.head())

#### Calculate Correlations

In [ ]:
# Extract age and item columns for correlation
age_for_corr = df_correlation_final['age']
hoff_items_for_corr = df_correlation_final[[col for col in df_correlation_final.columns if 'Scenario2_Hoff' in col]]
tia_items_for_corr = df_correlation_final[[col for col in df_correlation_final.columns if 'Scenario2_TIA' in col]]

# Calculate correlations
hoff_age_correlations = hoff_items_for_corr.corrwith(age_for_corr)
tia_age_correlations = tia_items_for_corr.corrwith(age_for_corr)

# Rename the index for plotting readability
hoff_age_correlations.index = [f'Hoff Item {i+1}' for i in range(len(hoff_age_correlations))]
tia_age_correlations.index = [f'TIA Item {i+1}' for i in range(len(tia_age_correlations))]

print("Correlation of Age with Hoff Scale Items:")
display(hoff_age_correlations.to_frame(name='Correlation with Age'))

print("\nCorrelation of Age with TIA Scale Items:")
display(tia_age_correlations.to_frame(name='Correlation with Age'))

### Factor Analysis for altered Hoffman and TIA Scales (Scenario 2)

In [ ]:
# Exploratory Factor Analysis (EFA) for Scenario 2
# Using the updated df_hoff_scenario2_reversed (Items 1, 2, 5, 7 reversed)
print("\n--- EFA for Scenario 2 (Items 1,2,5,7 Reversed) ---")
kmo_all_s2, kmo_model_s2 = calculate_kmo(df_hoff_scenario2_reversed)
bartlett_p_s2 = calculate_bartlett_sphericity(df_hoff_scenario2_reversed)[1]
print(f"KMO: {kmo_model_s2:.3f}, Bartlett p-value: {bartlett_p_s2:.3f}")

fa_s2 = FactorAnalyzer(rotation='varimax')
fa_s2.fit(df_hoff_scenario2_reversed)
ev_s2, v_s2 = fa_s2.get_eigenvalues()
print(f"Eigenvalues: {ev_s2}")

# Display Loadings
loadings_s2 = pd.DataFrame(fa_s2.loadings_, index=df_hoff_scenario2_reversed.columns, columns=[f'Factor {i+1}' for i in range(fa_s2.loadings_.shape[1])])
display(loadings_s2)

In [ ]:
# Plot the scree plot to determine the number of factors for the altered Hoffman scale
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(ev) + 1), ev, marker='o')
plt.axhline(1, color='red', linestyle='--', label='Eigenvalue = 1')
plt.title('Scree Plot for Hoff Scale Factor Analysis')
plt.xlabel('Factor Number')
plt.ylabel('Eigenvalue')
plt.grid(True)
plt.legend()
plt.show()

# Based on the scree plot, re-run Factor Analysis with the chosen number of factors
n_factors_hoff = sum(ev > 1) # Number of factors with eigenvalue > 1 (Kaiser Criterion)
if n_factors_hoff == 0: # Handle case where no eigenvalue is > 1
    n_factors_hoff = 1

fa_hoff_final = FactorAnalyzer(n_factors=n_factors_hoff, rotation='varimax')
fa_hoff_final.fit(df_hoff_scenario2_reversed)

print(f"\nUsing {n_factors_hoff} factor(s) for Hoff Scale Factor Analysis.")
print("Factor Loadings for Hoff Scale:")
hoff_loadings = pd.DataFrame(fa_hoff_final.loadings_, index=df_hoff_scenario2_reversed.columns)
display(hoff_loadings)

print("Factor Variance for Hoff Scale:")
display(pd.DataFrame(fa_hoff_final.get_factor_variance(), index=['SS Loadings', 'Proportion Var', 'Cumulative Var'], columns=[f'Factor {i+1}' for i in range(n_factors_hoff)]))


In [ ]:
# Perform Factor Analysis for TIA Scale
print("\n--- Factor Analysis for TIA Scale ---")
# Check for suitability of Factor Analysis using KMO and Bartlett's Test
kmo_all_tia, kmo_model_tia = calculate_kmo(df_tia_scenario2_reversed)
print(f"KMO Test: {kmo_model_tia:.3f}") # A value > 0.6 is generally considered good
bartlett_p_value_tia = calculate_bartlett_sphericity(df_tia_scenario2_reversed)[1]
print(f"Bartlett's Test p-value: {bartlett_p_value_tia:.3f}") # A p-value < 0.05 suggests data is suitable

# Perform Factor Analysis with a reasonable number of factors
fa_tia = FactorAnalyzer(n_factors=df_tia_scenario2_reversed.shape[1], rotation=None)
fa_tia.fit(df_tia_scenario2_reversed)

# Get eigenvalues
ev_tia, v_tia = fa_tia.get_eigenvalues()
print(f"Eigenvalues: {ev_tia}")


In [ ]:
# Plot the scree plot to determine the number of factors for TIA scale
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(ev_tia) + 1), ev_tia, marker='o')
plt.axhline(1, color='red', linestyle='--', label='Eigenvalue = 1')
plt.title('Scree Plot for TIA Scale Factor Analysis')
plt.xlabel('Factor Number')
plt.ylabel('Eigenvalue')
plt.grid(True)
plt.legend()
plt.show()

# Based on the scree plot, re-run Factor Analysis with the chosen number of factors
n_factors_tia = sum(ev_tia > 1) # Number of factors with eigenvalue > 1 (Kaiser Criterion)
if n_factors_tia == 0: # Handle case where no eigenvalue is > 1
    n_factors_tia = 1

fa_tia_final = FactorAnalyzer(n_factors=n_factors_tia, rotation='varimax')
fa_tia_final.fit(df_tia_scenario2_reversed)

print(f"\nUsing {n_factors_tia} factor(s) for TIA Scale Factor Analysis.")
print("Factor Loadings for TIA Scale:")
tia_loadings = pd.DataFrame(fa_tia_final.loadings_, index=df_tia_scenario2_reversed.columns)
display(tia_loadings)

print("Factor Variance for TIA Scale:")
display(pd.DataFrame(fa_tia_final.get_factor_variance(), index=['SS Loadings', 'Proportion Var', 'Cumulative Var'], columns=[f'Factor {i+1}' for i in range(n_factors_tia)]))


###Paired t-test for TIA Scale

Given that the survey likely involves the same participants responding to both scenarios, a paired t-test is appropriate to detect possible different response patterns in the TIA or altered Hoffman scale.



In [ ]:
import pandas as pd
from scipy import stats

# Re-define df_tia_reversed for Scenario 1 (to ensure consistent cleaning and availability as apparently some participant answered only in one scenario)
tia_cols_s1 = [col for col in df_cleaned_headers.columns if 'Scenario1_TIA' in col]
df_tia_s1_raw = df_cleaned_headers[tia_cols_s1].copy()

# Preprocessing: Convert to numeric and keep only 1-5 range
for col in df_tia_s1_raw.columns:
    df_tia_s1_raw[col] = pd.to_numeric(df_tia_s1_raw[col], errors='coerce')

# Drop rows that are entirely NaN
df_tia_s1_cleaned = df_tia_s1_raw.dropna(how='all')

# Filter for valid Likert range 1-5
df_tia_s1_cleaned = df_tia_s1_cleaned[(df_tia_s1_cleaned >= 1).all(axis=1) & (df_tia_s1_cleaned <= 5).all(axis=1)]

# Rename columns for consistency (e.g., TIA_1, TIA_2, etc.)
df_tia_s1_renamed = df_tia_s1_cleaned.copy()
df_tia_s1_renamed.columns = [f'TIA_{i+1}' for i in range(len(df_tia_s1_renamed.columns))]

# Reverse-code items TIA_6 through TIA_11
reverse_cols_tia_s1 = ['TIA_6', 'TIA_7', 'TIA_8', 'TIA_9', 'TIA_10', 'TIA_11']
df_tia_s1_reversed = df_tia_s1_renamed.copy()
for col in reverse_cols_tia_s1:
    if col in df_tia_s1_reversed.columns:
        df_tia_s1_reversed[col] = 6 - df_tia_s1_reversed[col]

# Align the two dataframes based on their indices to ensure a paired comparison
# This means keeping only the participants who have valid data in **both** scenarios
common_indices_tia = df_tia_s1_reversed.index.intersection(df_tia_scenario2_reversed.index)

df_tia_s1_aligned = df_tia_s1_reversed.loc[common_indices_tia]
df_tia_s2_aligned = df_tia_scenario2_reversed.loc[common_indices_tia]

# Calculate the mean TIA score for each participant in Scenario 1 from the aligned data
mean_tia_s1 = df_tia_s1_aligned.mean(axis=1)

# Calculate the mean TIA score for each participant in Scenario 2 from the aligned data
mean_tia_s2 = df_tia_s2_aligned.mean(axis=1)

# Perform the paired t-test
t_statistic_tia, p_value_tia = stats.ttest_rel(mean_tia_s1, mean_tia_s2)

print(f"Mean TIA Score (Scenario 1): {mean_tia_s1.mean():.3f}")
print(f"Mean TIA Score (Scenario 2): {mean_tia_s2.mean():.3f}")
print(f"\nPaired t-test results for TIA Scale (Scenario 1 vs. Scenario 2):\n")
print(f"t-statistic: {t_statistic_tia:.3f}")
print(f"p-value: {p_value_tia:.3f}")

# Set significance value alpha=0.05 (significant); change to 0.01 for more conservative measure
alpha = 0.05
if p_value_tia < alpha:
    print(f"\nSince the p-value ({p_value_tia:.3f}) is less than the significance level ({alpha}), we reject the null hypothesis.\nThere is a statistically significant difference between the TIA scale means in Scenario 1 and Scenario 2.")
else:
    print(f"\nSince the p-value ({p_value_tia:.3f}) is greater than the significance level ({alpha}), we fail to reject the null hypothesis.\nThere is no statistically significant difference between the TIA scale means in Scenario 1 and Scenario 2.")

### Paired t-test altered Hoffman Scale



In [ ]:
from scipy import stats

# Calculate the mean Hoffman score for each participant in Scenario 1
mean_hoff_s1 = df_quest_reversed.mean(axis=1)

# Calculate the mean Hoffman score for each participant in Scenario 2
mean_hoff_s2 = df_hoff_scenario2_reversed.mean(axis=1)

# As per the context, df_quest_reversed (Scenario 1) has 59 rows and df_hoff_scenario2_reversed (Scenario 2) also has 59 rows.
# We assume the indices are aligned due to the sequential cleaning process.

# Perform the paired t-test
t_statistic, p_value = stats.ttest_rel(mean_hoff_s1, mean_hoff_s2)

print(f"Mean Hoffman Score (Scenario 1): {mean_hoff_s1.mean():.3f}")
print(f"Mean Hoffman Score (Scenario 2): {mean_hoff_s2.mean():.3f}")
print(f"\nPaired t-test results for Hoffman Scale (Scenario 1 vs. Scenario 2):\n")
print(f"t-statistic: {t_statistic:.3f}")
print(f"p-value: {p_value:.3f}")

# Set signifance level to 0.05 (significant); change to \alpha=0.01 (very significant) to be more conservative
alpha = 0.05
if p_value < alpha:
    print(f"\nSince the p-value ({p_value:.3f}) is less than the significance level ({alpha}), we reject the null hypothesis.\nThere is a statistically significant difference between the Hoffman scale means in Scenario 1 and Scenario 2.")
else:
    print(f"\nSince the p-value ({p_value:.3f}) is greater than the significance level ({alpha}), we fail to reject the null hypothesis.\nThere is no statistically significant difference between the Hoffman scale means in Scenario 1 and Scenario 2.")

In [ ]:
# Perform Factor Analysis for TIA Scale
print("\n--- Factor Analysis for TIA Scale ---")
# Check for suitability of Factor Analysis using KMO and Bartlett's Test
kmo_all_tia, kmo_model_tia = calculate_kmo(df_tia_scenario2_reversed)
print(f"KMO Test: {kmo_model_tia:.3f}") # A value > 0.6 is generally considered good
bartlett_p_value_tia = calculate_bartlett_sphericity(df_tia_scenario2_reversed)[1]
print(f"Bartlett's Test p-value: {bartlett_p_value_tia:.3f}") # A p-value < 0.05 suggests data is suitable

# Perform Factor Analysis with a reasonable number of factors
fa_tia = FactorAnalyzer(n_factors=df_tia_scenario2_reversed.shape[1], rotation=None)
fa_tia.fit(df_tia_scenario2_reversed)

# Get eigenvalues
ev_tia, v_tia = fa_tia.get_eigenvalues()
print(f"Eigenvalues: {ev_tia}")


In [ ]:
# Plot the scree plot to determine the number of factors for TIA scale
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(ev_tia) + 1), ev_tia, marker='o')
plt.axhline(1, color='red', linestyle='--', label='Eigenvalue = 1')
plt.title('Scree Plot for TIA Scale Factor Analysis')
plt.xlabel('Factor Number')
plt.ylabel('Eigenvalue')
plt.grid(True)
plt.legend()
plt.show()

# Based on the scree plot, re-run Factor Analysis with the chosen number of factors
n_factors_tia = sum(ev_tia > 1) # Number of factors with eigenvalue > 1 (Kaiser Criterion)
if n_factors_tia == 0: # Handle case where no eigenvalue is > 1
    n_factors_tia = 1

fa_tia_final = FactorAnalyzer(n_factors=n_factors_tia, rotation='varimax')
fa_tia_final.fit(df_tia_scenario2_reversed)

print(f"\nUsing {n_factors_tia} factor(s) for TIA Scale Factor Analysis.")
print("Factor Loadings for TIA Scale:")
tia_loadings = pd.DataFrame(fa_tia_final.loadings_, index=df_tia_scenario2_reversed.columns)
display(tia_loadings)

print("Factor Variance for TIA Scale:")
display(pd.DataFrame(fa_tia_final.get_factor_variance(), index=['SS Loadings', 'Proportion Var', 'Cumulative Var'], columns=[f'Factor {i+1}' for i in range(n_factors_tia)]))


In [ ]:
from scipy import stats

# Identify the unique age categories in the combined_df_factors
age_categories = sorted(combined_df_factors['age'].unique())
print(f"Age categories identified: {age_categories}")

# List of factor scores to test
factor_score_cols = [
    'Hoffman_S1_Factor1',
    'Hoffman_S1_Factor2',
    'Hoffman_S1_Factor3',
    'Hoffman_S2_Factor1',
    'Hoffman_S2_Factor2',
    'TIA_S2_Factor1',
    'TIA_S2_Factor2'
]

print("\n--- Kruskal-Wallis H-test Results (Factor Scores vs. Age Categories) ---")
alph_level = 0.05

for factor_col in factor_score_cols:
    # Prepare data for Kruskal-Wallis: list of arrays, one for each age group's factor scores
    data_for_kruskal = [combined_df_factors[combined_df_factors['age'] == cat][factor_col].dropna().values for cat in age_categories]

    # Ensure there's at least one non-empty group to perform the test
    valid_data_for_kruskal = [data for data in data_for_kruskal if len(data) > 0]

    if len(valid_data_for_kruskal) < 2: # Kruskal-Wallis requires at least 2 groups
        print(f"\nSkipping Kruskal-Wallis for {factor_col}: Not enough valid age groups with data.")
        continue

    # Perform the Kruskal-Wallis H-test
    h_statistic, p_value = stats.kruskal(*valid_data_for_kruskal)

    print(f"\n{factor_col}:")
    print(f"  H-statistic: {h_statistic:.3f}")
    print(f"  p-value: {p_value:.3f}")
    if p_value < alph_level:
        print(f"  Result: Statistically significant difference across age categories (p < {alph_level})")
    else:
        print(f"  Result: No statistically significant difference across age categories (p >= {alph_level})")


### Post-Hoc Analysis for Hoffman S1 Factor 2 vs. Age Categories (Dunn's Test)

Since the Kruskal-Wallis H-test showed a statistically significant difference for "Hoffman_S1_Factor2" across age categories (p-value = 0.048), we perform a post-hoc test to determine which specific age groups differ from each other. Dunn's test (with Bonferroni correction for multiple comparisons).

In [ ]:
# Install scikit-posthocs if not already installed
!pip install scikit-posthocs

import scikit_posthocs as sp

# Prepare data for Dunn's test
# We need the 'Hoffman_S1_Factor2' scores and their corresponding 'age' categories.
# Drop any rows with NaN values in either column to ensure clean comparison.
plot_df = combined_df_factors[['age', 'Hoffman_S1_Factor2']].dropna()

# Perform Dunn's test (using 'auto' method for p-value correction, typically Bonferroni or Sidak)
# The 'by' argument specifies the grouping variable ('age')
dunn_test_results = sp.posthoc_dunn(plot_df, val_col='Hoffman_S1_Factor2', group_col='age', p_adjust='bonferroni')

print("\n--- Dunn's Post-Hoc Test Results for Hoffman S1 Factor 2 vs. Age Categories (Bonferroni corrected) ---")
display(dunn_test_results)

# Interpretation guidance:
print("\nInterpretation: Look for p-values less than 0.05. These indicate a statistically significant difference between the two compared age categories.")


### Visualizing Hoffman S1 Factor 2 by Age Category

To further understand the nature of the significant difference found by the Kruskal-Wallis and Dunn's test (if any significant pairs are found), a box plot can visually represent the distribution of `Hoffman_S1_Factor2` scores across each age category.

### Mann-Whitney U Test for Hoffman S1 Factor 2: Age Category 4.0 vs. 5.0

Since Dunn's post-hoc test indicated a significant difference between age categories 4.0 and 5.0 for `Hoffman_S1_Factor2`, we can perform a direct Mann-Whitney U test to confirm this pairwise comparison and report its statistics.

In [ ]:
from scipy import stats

# Extract Hoffman_S1_Factor2 scores for age group 4.0
age_group_4 = plot_df[plot_df['age'] == 4.0]['Hoffman_S1_Factor2'].dropna()

# Extract Hoffman_S1_Factor2 scores for age group 5.0
age_group_5 = plot_df[plot_df['age'] == 5.0]['Hoffman_S1_Factor2'].dropna()

# Perform Mann-Whitney U test
statistic, p_value = stats.mannwhitneyu(age_group_4, age_group_5, alternative='two-sided')

print(f"--- Mann-Whitney U Test Results for Hoffman S1 Factor 2 (Age 4.0 vs. Age 5.0) ---")
print(f"Mean Hoffman S1 Factor 2 (Age 4.0): {age_group_4.mean():.3f}")
print(f"Mean Hoffman S1 Factor 2 (Age 5.0): {age_group_5.mean():.3f}")
print(f"Mann-Whitney U Statistic: {statistic:.3f}")
print(f"P-value: {p_value:.3f}")

alpha = 0.05
if p_value < alpha:
    print(f"Result: Statistically significant difference between Age 4.0 and Age 5.0 (p < {alpha})")
else:
    print(f"Result: No statistically significant difference between Age 4.0 and Age 5.0 (p >= {alpha})")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(x='age', y='Hoffman_S1_Factor2', data=plot_df, palette='viridis')
plt.title('Distribution of Hoffman S1 Factor 2 Scores Across Age Categories')
plt.xlabel('Age Category (1-5)')
plt.ylabel('Hoffman S1 Factor 2 Score')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


### Survey Demographics


Participant Counts by Age Group (1-5)

In [ ]:
import pandas as pd

# Extract the 'age' column from the original data frame
age_raw = df['age'].copy()
age_cleaned = age_raw.iloc[2:].copy()
age_cleaned = pd.to_numeric(age_cleaned, errors='coerce')
age_cleaned = age_cleaned.dropna()

# Convert to DataFrame for consistent handling and count
age_df_for_counts = age_cleaned.to_frame(name='age')

# Count participants per age group
if not age_df_for_counts.empty:
    age_counts = age_df_for_counts['age'].value_counts().sort_index()
    print("Number of participants per age group:")
    display(age_counts.to_frame(name='Count'))
else:
    print("No valid age data found after cleaning.")

Participant Counts by Sea Experience Group (1-5)

In [ ]:
import pandas as pd

# Extract the 'sea_service' column from the original DataFrame and clean it
# This follows the cleaning logic used for age elsewhere in the notebook
sea_service_raw = df['sea_service'].copy()
sea_service_cleaned = sea_service_raw.iloc[2:].copy() # Drop the first two header rows
sea_service_cleaned = pd.to_numeric(sea_service_cleaned, errors='coerce')
sea_service_cleaned = sea_service_cleaned.dropna() # Drop NaN values from sea_service

# Convert to DataFrame for consistent handling and count
sea_service_df_for_counts = sea_service_cleaned.to_frame(name='sea_service')

# Count participants per sea experience group
if not sea_service_df_for_counts.empty:
    sea_service_counts = sea_service_df_for_counts['sea_service'].value_counts().sort_index()
    print("Number of participants per sea experience group:")
    display(sea_service_counts.to_frame(name='Count'))
else:
    print("No valid sea experience data found after cleaning.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read the 'rank_1' column from the main DataFrame
rank_1_raw = df['rank_1'].copy()

# Define a mapping for standardizing rank variations
rank_mapping = {
    'captain': 'Captain',
    'master': 'Captain',
    'kapitein': 'Captain',
    'cpt': 'Captain',
    'captain /riverpilot': 'Captain',
    'master all ships': 'Captain',
    'master dpo': 'Captain',
    'retired master roro vessels 40 years at sea': 'Captain',
    'oow': 'Officer of the Watch',
    'oow/ bel navy nav instructor': 'Officer of the Watch',
    'officer of the watch': 'Officer of the Watch',
    '2nd officer': 'Second Officer',
    'second officer': 'Second Officer',
    '2nd off': 'Second Officer',
    '2 officer': 'Second Officer',
    '2/0': 'Second Officer',
    'chief officer': 'Chief Officer',
    '3rd officer': 'Third Officer',
    'third officer': 'Third Officer',
    'engineer': 'Engineer',
    'chief engineer': 'Chief Engineer',
    '2nd engineer': 'Second Engineer',
    'third engineer': 'Third Engineer',
    'cadet': 'Cadet',
    'deck cadet': 'Cadet',
    'engine cadet': 'Cadet',
    'student': 'Cadet', # Group 'student' with Cadet
    'deckhand': 'Deckhand',
    'pilot': 'Pilot',
    'sea pilot': 'Pilot',
    'retired pilot': 'Pilot',
    'rivierloods op rust': 'Pilot',
    'pilot ( retired 3y )': 'Pilot',
    'pilot (retired)': 'Pilot',
    'chief mate': 'Chief Mate',
    '2nd mate': 'Second Mate',
    'navigation officer': 'Navigation Officer',
    'head of navigation': 'Navigation Officer',
    'xo (second in command)': 'Second in Command',
    'nan': 'Other', # Treat NaN values as Other
    'what is your highest rank or your current position? (oow, captain, etc.)? - rank/position': 'Other', # Treat long descriptive text as Other
    'other': 'Other'
}

# Standardize capitalization and remove leading/trailing spaces for better matching
# Also handle potential NaN values by converting to string before lowercasing
rank_1_cleaned = rank_1_raw.astype(str).str.lower().str.strip()

# Apply the mapping. For entries not in the map, default to 'Other'.
df['rank_1_standardized'] = rank_1_cleaned.apply(lambda x: rank_mapping.get(x, 'Other'))

print("Original 'rank_1' values (first 10):")
display(rank_1_raw.head(10))

print("\nStandardized 'rank_1' values (first 10):")
display(df['rank_1_standardized'].head(10))

# Display the distribution of the standardized 'rank_1' column
standardized_rank_counts = df['rank_1_standardized'].value_counts()

print("\nDistribution of standardized 'rank_1' column:")
display(standardized_rank_counts.to_frame(name='Count')) # Display as a DataFrame for better readability

# Plot the result in a bar chart
plt.figure(figsize=(12, 7))
sns.barplot(x=standardized_rank_counts.index, y=standardized_rank_counts.values, palette='viridis', hue=standardized_rank_counts.index, legend=False)
plt.title('Distribution of Standardized Ranks')
plt.xlabel('Standardized Rank')
plt.ylabel('Number of Participants')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

Survey Completion Time

In [ ]:
import pandas as pd

# Ensure the dates are in datetime format
duration_df = df.iloc[2:].copy()
duration_df['StartDate'] = pd.to_datetime(duration_df['StartDate'])
duration_df['EndDate'] = pd.to_datetime(duration_df['EndDate'])

# Calculate duration in minutes
duration_df['duration_min'] = (duration_df['EndDate'] - duration_df['StartDate']).dt.total_seconds() / 60

# Calculate mean and standard deviation
mean_duration = duration_df['duration_min'].mean()
std_duration = duration_df['duration_min'].std()

print(f'Mean completion time: {mean_duration:.2f} minutes')
print(f'Standard deviation: {std_duration:.2f} minutes')

# Optional: Display top 5 rows to verify
display(duration_df[['StartDate', 'EndDate', 'duration_min']].head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the visual style
sns.set_theme(style="whitegrid")

plt.figure(figsize=(12, 6))
# Create a histogram of the duration in minutes
sns.histplot(duration_df['duration_min'], bins=50, kde=True, color='skyblue')

plt.title('Distribution of Survey Completion Times')
plt.xlabel('Duration (minutes)')
plt.ylabel('Frequency')

# Given the high standard deviation, let's also show a version focused on the bulk of the data
plt.show()

# Secondary plot for better visibility of the majority of participants (excluding extreme outliers)
plt.figure(figsize=(12, 6))
sns.histplot(duration_df[duration_df['duration_min'] < 120]['duration_min'], bins=30, kde=True, color='salmon')
plt.title('Distribution of Completion Times (Focussed on < 120 minutes)')
plt.xlabel('Duration (minutes)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
under_60_count = (duration_df['duration_min'] < 60).sum()
total_count = len(duration_df)
percentage_under_60 = (under_60_count / total_count) * 100

print(f'Number of participants under 60 minutes: {under_60_count}')
print(f'Total participants: {total_count}')
print(f'Percentage under 60 minutes: {percentage_under_60:.2f}%')

In [ ]:
# Display the distribution of the standardized 'rank_1' column
standardized_rank_counts = df['rank_1_standardized'].value_counts()

print("Distribution of standardized 'rank_1' column:")
display(standardized_rank_counts)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 7))
sns.barplot(x=standardized_rank_counts.index, y=standardized_rank_counts.values, palette='viridis', hue=standardized_rank_counts.index, legend=False)
plt.title('Distribution of Standardized Ranks')
plt.xlabel('Standardized Rank')
plt.ylabel('Number of Participants')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Filter out the 'Other' category from the standardized ranks
filtered_standardized_rank_counts = df[df['rank_1_standardized'] != 'Other']['rank_1_standardized'].value_counts()

print("Distribution of standardized 'rank_1' column (excluding 'Other'):")
display(filtered_standardized_rank_counts.to_frame(name='Count'))

# Plot the filtered result in a bar chart
plt.figure(figsize=(12, 7))
sns.barplot(x=filtered_standardized_rank_counts.index, y=filtered_standardized_rank_counts.values, palette='viridis', hue=filtered_standardized_rank_counts.index, legend=False)
plt.title('Sea Rank Distribution')
plt.xlabel('Standardized Rank')
plt.ylabel('Number of Participants')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
rank_1_counts = df['rank_1'].value_counts()

print("Distribution of 'rank_1' column:")
display(rank_1_counts)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 7))
sns.barplot(x=rank_1_counts.index, y=rank_1_counts.values, palette='viridis', hue=rank_1_counts.index, legend=False)
plt.title('Distribution of Ranks (rank_1)')
plt.xlabel('Rank')
plt.ylabel('Number of Participants')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.barplot(x=sea_service_counts.index, y=sea_service_counts.values, palette='viridis', hue=sea_service_counts.index, legend=False)
plt.title('Sea Rank Distribution')
plt.xlabel('Sea Experience Rank (1-5)')
plt.ylabel('Number of Participants')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()